In [0]:
%sql
-- Análisis de Cancelaciones en Hotel - DataAndino
-- // SQL (sintaxis Databricks) | Tabla base: tb_reservas 
-- // Resultados verificados sobre hotel_bookings.csv (119,390 filas)
USE CATALOG workspace;


In [0]:
%sql
USE SCHEMA hotel_db;

In [0]:
%sql
SELECT COUNT(*) FROM tb_reservas_raw;

In [0]:
%sql
SELECT COUNT(*) AS total,
       COUNT(arrival_date) AS con_fecha_llegada,
       COUNT(reservation_status_date) AS con_fecha_estado
FROM tb_reservas;

In [0]:
%sql
-- EJERCICIO 1 (CREATE / INSERT tabla)

CREATE OR REPLACE TABLE tb_reservas_prueba
USING DELTA AS
SELECT *
FROM tb_reservas
WHERE 1 = 0;

INSERT INTO tb_reservas_prueba
SELECT *
FROM tb_reservas
WHERE hotel = 'City Hotel'
  AND is_canceled = 1;

In [0]:
%sql
SELECT COUNT(*) AS filas
FROM tb_reservas_prueba;

In [0]:
%sql
-- EJERCICIO 2 (UPDATE)
UPDATE tb_reservas_prueba
SET babies = 0;
-- CONCLUSIÓN: Se corrigió el error de captura de babies en la tabla de prueba

In [0]:
%sql
-- EJERCICIO 3 (CAST / CONCAT)
SELECT
    arrival_date_year,
    arrival_date_month,
    arrival_date_day_of_month,
    CONCAT(CAST(arrival_date_day_of_month AS STRING), '-',
           arrival_date_month, '-',
           CAST(arrival_date_year AS STRING)) AS arrival_date_string
FROM tb_reservas
LIMIT 10;
-- CONCLUSIÓN: La fecha de llegada queda legible (ej. 1-July-2015) 

In [0]:
%sql
-- EJERCICIO 4 (LIKE)
SELECT *
FROM tb_reservas
WHERE distribution_channel LIKE 'T%'
LIMIT 10;


In [0]:
%sql
-- EJERCICIO 5 (COUNT / SUM / AVG)
SELECT
    COUNT(*) AS total_reservas,
    SUM(total_of_special_requests) AS total_solicitudes_especiales,
    ROUND(AVG(adr), 2) AS adr_promedio
FROM tb_reservas;
-- CONCLUSIÓN: Hay 119,390 reservas, 68,215 solicitudes especiales y un precio promedio de USD 101.83 por noche.

In [0]:
%sql
-- EJERCICIO 6 (DELETE)
DELETE FROM tb_reservas_prueba
WHERE adr = 0.0;
-- CONCLUSIÓN: Se eliminaron 129 pruebas del sistema (adr = 0) de la tabla de prueba, asi se deja solo reservas reales.

In [0]:
%sql
-- EJERCICIO 7
SELECT
    distribution_channel,
    COUNT(*) AS cantidad_reservas
FROM tb_reservas
GROUP BY distribution_channel
ORDER BY cantidad_reservas DESC;

--CONCLUSIÓN: TA/TO (97,870) concentra más del 80% de las reservas. En ese sentido, el contrato con la agencia tiene un impacto financiero grande.

In [0]:
%sql
-- EJERCICIO 08
SELECT
    adr,
    CASE
        WHEN adr < 50 THEN 'Económico'
        WHEN adr BETWEEN 50 AND 150 THEN 'Estándar'
        ELSE 'Premium'
    END AS categoria_precio
FROM tb_reservas
LIMIT 20;
-- CONCLUSIÓN: El 76% de las reservas es Estándar (91,293); Premium suma 16,508 y Económico 11,589.

In [0]:
%sql
-- EJERCICIO 09 (DATE_SUB)
SELECT
    arrival_date,
    lead_time,
    DATE_SUB(arrival_date, CAST(lead_time AS INT)) AS booking_date
FROM tb_reservas
LIMIT 10;
-- CONCLUSIÓN: Con booking_date se sabe cuándo reserva el cliente, no solo cuándo llega. Con esto se puede analizar la demanda

In [0]:
%sql
-- EJERCICIO 10
SELECT
    lead_time,
    CEIL(lead_time / 30.0) AS meses_anticipacion_techo
FROM tb_reservas
LIMIT 10;

-- objetivo: convertir los días de anticipación en meses enteros, esto es útil para agrupar reservas por meses de anticipación.

In [0]:
%sql
-- EJERCICIO 11 
SELECT
    COUNT(*) AS reservas_canceladas_agencia,
    COUNT(*) * 1.5 AS perdida_usd
FROM tb_reservas
WHERE distribution_channel = 'TA/TO'
  AND is_canceled = 1;
-- Resultado: 40,152 reservas | USD 60,228
-- CONCLUSIÓN: Se pagaron USD 60,228 a la agencia por 40,152 reservas que se cancelaron.

In [0]:
%sql
-- EJERCICIO 12 (costo de cancelar tarde)
SELECT
    COUNT(*) AS cancelaciones_tardias,
    COUNT(*) * 120 AS perdida_total_usd
FROM tb_reservas
WHERE is_canceled = 1
  AND DATEDIFF(arrival_date, reservation_status_date) < 3;
-- Resultado: 3,204 reservas | USD 384,480
-- CONCLUSIÓN: Las cancelaciones de última hora cuestan USD 384,480, más de 6 veces lo pagado a la agencia por reservas canceladas.

In [0]:
%sql
CREATE OR REPLACE TABLE tb_paises (
    cod_pais STRING,
    nombre_pais STRING
);

In [0]:
%sql
INSERT INTO tb_paises
VALUES ('PRT', 'Portugal'),
       ('GBR', 'Reino Unido'),
       ('ESP', 'España'),
       ('FRA', 'Francia'),
       ('ITA', 'Italia'),
       ('DEU', 'Alemania'),
       ('IRL', 'Irlanda'),
       ('BEL', 'Bélgica'),
       ('BRA', 'Brasil'),
       ('NLD', 'Países Bajos'),
       ('USA', 'Estados Unidos'),
       ('CHE', 'Suiza');

In [0]:
%sql
CREATE OR REPLACE TABLE tb_vendedores (
    id_vendedor INT,
    nombre_vendedor STRING
);

In [0]:
%sql
INSERT INTO tb_vendedores
VALUES (1, 'Ana Torres'),
       (2, 'Luis Rojas'),
       (3, 'Carla Mendoza'),
       (4, 'Pedro Salas'),
       (5, 'Rosa Quispe'),
       (6, 'Jorge Vega');

In [0]:
%sql
CREATE OR REPLACE TABLE tb_facturas (
    id_reserva INT,
    monto_facturado DOUBLE
);

In [0]:
%sql
INSERT INTO tb_facturas
SELECT
    id_reserva,
    ROUND(adr * (stays_in_weekend_nights + stays_in_week_nights), 2)
FROM tb_reservas
WHERE is_canceled = 0
  AND id_reserva <= 500;

In [0]:
%sql
INSERT INTO tb_facturas
VALUES (999999, 150.0);

In [0]:
%sql
SELECT
    r.is_canceled,
    p.nombre_pais
FROM tb_reservas r
LEFT JOIN tb_paises p
       ON r.country = p.cod_pais
LIMIT 100;

-- CONCLUSIÓN: El LEFT JOIN conserva todas las reservas. Las que no tienen país en tb_paises salen con nombre_pais NULL.

In [0]:
%sql
-- EJERCICIO 14 (INNER JOIN + GROUP BY)
SELECT
    p.nombre_pais,
    COUNT(*) AS reservas_canceladas
FROM tb_reservas r
INNER JOIN tb_paises p
        ON r.country = p.cod_pais
WHERE r.is_canceled = 1
GROUP BY p.nombre_pais
ORDER BY reservas_canceladas DESC
LIMIT 5;
-- Resultado: Portugal 27,519 | Reino Unido 2,453 | España 2,177 |  Francia 1,934 | Italia 1,333
-- CONCLUSIÓN: Portugal contiene más del 60% de las cancelaciones.

In [0]:
%sql
-- EJERCICIO 15 (FULL OUTER JOIN)
SELECT
    r.distribution_channel,
    v.nombre_vendedor
FROM tb_reservas r
FULL OUTER JOIN tb_vendedores v
             ON r.id_vendedor = v.id_vendedor
WHERE r.id_vendedor IS NULL
   OR v.id_vendedor IS NULL;

-- Resultado: 2,387 reservas sin vendedor
-- CONCLUSIÓN: El FULL OUTER JOIN muestra ambos lados sin coincidencia: reservas
-- sin vendedor (vendedor NULL) y vendedores sin reservas (canal NULL).

In [0]:
%sql
-- EJERCICIO 16 (RIGHT JOIN)
SELECT
    f.id_reserva,
    f.monto_facturado,
    r.hotel
FROM tb_reservas r
RIGHT JOIN tb_facturas f
        ON r.id_reserva = f.id_reserva
ORDER BY r.hotel;


-- CONCLUSIÓN: Aparecen todas las facturas emitidas. La que tiene hotel NULL no tiene reserva asociada.